# Analysis of the Allegheny County Jail Oversight Board Meeting Minutes

- Contributor: Anna Ringwood
- AI Acknowledgements: The following code was written with some assistance from GitHub Copilot.

## Text Preprocessing — Processing Text from JOB Minutes

This notebook contains code for ingesting the scraped text files and beginning to further extract only relevant text information.

### Import libraries and raw document data:

In [266]:
import csv
import time
from pathlib import Path
from urllib.parse import urlparse
import html
import re
import numpy as np

In [212]:
# Set up paths, similar to how we did in the preceding file
BASE = Path("..").resolve()
OUT_DIR = BASE / "Data" / "Text" # where the raw text files are kept

In [224]:
raw_docs = []
doc_names = []

# This section's code taken from last block of preceding file with minor modifications to variable names
for path in sorted(OUT_DIR.glob("*.txt")):
    raw_docs.append(path.read_text(encoding="utf-8"))
    doc_names.append(path.name)

print(f"{len(raw_docs)} document(s) → lists `documents` (text) and `doc_names` (filenames)")

156 document(s) → lists `documents` (text) and `doc_names` (filenames)


In [193]:
# Create a helper function to convert document names to their indices when needed (more useful for development than production)
def doc_name_to_idx(name):
    assert name in doc_names, "Document name not found"
    for i, doc_name in enumerate(doc_names):
        if doc_name == name:
            print(f"Document {i} has the title {name}.")

# Test function
doc_name_to_idx("2021_10_0_JOB_Minutes.txt") # returns index 110
doc_names[110]

Document 90 has the title 2021_10_0_JOB_Minutes.txt.


'2022_5_0_JOB_Minutes.txt'

### Remove characters that are ill-handled by `spaCy`:

Before tokenizing with `spaCy`, we'll address some HTML artifacts (e.g., `\xa0`). We'll also replace any instances of white space with a single space. Finally, based on prior exploration, private-use characters (e.g., bullet points from Word documents) get classified as NOUN or PROPN, which makes it difficult to filter them out post-`spaCy`-processing. Therefore, we'll remove those characters from the raw text as well.

In [ ]:
# unescaped_docs = []
# for doc in raw_docs:
#     temp_doc = html.unescape(doc) # unescape HTML entities; courtesy of MS Copilot
#     temp_doc = re.sub(r'\s+', ' ', temp_doc) # also from MS Copilot
#     unescaped_docs.append(temp_doc.strip())

In [225]:
# Check which weird characters are still present
weird_chars = set()
for doc in raw_docs:
    weird_chars.update([char for char in doc if ord(char) > 127]) # "ord(char) > 127" courtesy of Stack Overflow

print(weird_chars)

{'ć', 'ó', '½', 'é', '—', '‐', '♦', '…', '’', '©', 'Ɵ', '\xa0', '\uf0b7', 'ñ', '\uf0a7', '§', '“', '‘', '–', '•', '”'}


Some of the "weird" characters may very well change the document's meaning if removed (e.g., " ñ ", " ó ", " é "), so we'll keep those and remove all other characters.

In [226]:
# Remove all weird chars except for accented letters from weird_chars
remove_chars = {char for char in weird_chars if char not in {'ñ', 'ó', 'é', 'ć'}}

In [227]:
prelim_docs = []
for doc in raw_docs:
    temp_doc = doc.replace("\n", " ") # replace newlines with spaces
    for char in remove_chars:
        temp_doc = temp_doc.replace(char, " ") # replace each weird character with a space
    temp_doc = " ".join(temp_doc.split()) # replace multiple spaces with a single space - done last so as to not accidentally introduce multiple spaces
    prelim_docs.append(temp_doc)

In [228]:
print(f"Before newline removal: {raw_docs[115][:100]}")
print(f"After newline removal: {prelim_docs[115][:100]}")

Before newline removal: 1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
GALVIN REPORTING SERVICES
412-897-
After newline removal: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 GALVIN REPORTING SERVICES 412-897-


In [231]:
# Find where the first instance of the bullet character is in the original document to confirm that it was replaced
print(f"Before weird char removal: {raw_docs[1][(raw_docs[1].find("\uf0b7"))-20:(raw_docs[1].find("\uf0b7")+20)]}")
print(f"After weird char removal: {prelim_docs[1][(prelim_docs[1].find("sed the following: ")):(prelim_docs[1].find("Congratulated the")+len("Congratulated the"))]}")

Before weird char removal: sed the following: 
 Congratulated the 
After weird char removal: sed the following: Congratulated the


The newline characters have been replaced with spaces, there are no multi-spaces, and we've removed the unicode characters, so we can use spaCy now to conduct additional preprocessing.

### Remove unneeded transcription text:

In more recent years the minutes are just transcripts with line numbers and transcription company information on each page. We don't need this data, so we filter it out.

In [232]:
# Get indices of the documents that contain "GALVIN REPORTING SERVICES" a.k.a. the ones that contain transcription line numbers + company name we want to remove
transcription_junk = "1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 GALVIN REPORTING SERVICES 412-897-2010 -- 412-461-1838 (FAX)"
docs_w_transcription_junk = []

for i, doc in enumerate(prelim_docs):
    if "GALVIN REPORTING SERVICES" in doc:
        docs_w_transcription_junk.append(i)

In [233]:
# Remove all occurrences of the transcription junk from the documents
for i in docs_w_transcription_junk:
    prelim_docs[i] = prelim_docs[i].replace(transcription_junk, "")

In [234]:
[prelim_docs[i][:50] for i in docs_w_transcription_junk[:10]] # check that junk was removed - scan of the first 10 documents show that the removal was successful

[' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T']

### Remove end matter of minutes

Additionally, the transcribed documents include end matter with vocab glossaries and indices, which would both throw off our analysis and take up unnecessary processing power during lemmatization. We therefore looked for keywords that would indicate the end of the transcription (e.g., "adjournment"). After some trial-and-error, we discovered that all of the transcripted documents except one contained a notary certification at the end, after which the glossary and index appeared. Thus, for these docs we removed all the text after the certification header. For the final document that didn't follow this pattern, we manually trimmed the end matter.

In [235]:
one_occurrence_idxs = []
gt_one_occurrence_idxs = []
lt_one_occurrence_idxs = []
word_phrase = "C E R T I F I C A T E"

for i in docs_w_transcription_junk:
    
    if prelim_docs[i].upper().count(word_phrase) > 1:
        gt_one_occurrence_idxs.append(i)
    elif prelim_docs[i].upper().count(word_phrase) < 1:
        lt_one_occurrence_idxs.append(i)
    else:
        one_occurrence_idxs.append(i)

print(f"One instance ({len(one_occurrence_idxs)}): {one_occurrence_idxs}")
print(f"Less than one instance ({len(lt_one_occurrence_idxs)}): {lt_one_occurrence_idxs}")
print(f"More than one instance ({len(gt_one_occurrence_idxs)}): {gt_one_occurrence_idxs}")

One instance (43): [103, 104, 105, 115, 116, 117, 118, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155]
Less than one instance (1): [119]
More than one instance (0): []


In [236]:
doc_names[119] # upon manual inspection, we'll remove everything after "I, Diane  G. Galvin , a court  reporter"

'2023_2_0_JOB_Minutes.txt'

In [237]:
# Remove all text after "C E R T I F I C A T E" in the documents
for i in one_occurrence_idxs:
    cert_idx = prelim_docs[i].upper().find(word_phrase)
    prelim_docs[i] = prelim_docs[i][:cert_idx] # keep everything up to but not including the word "C E R T I F I C A T E "

# For document 119, remove everything after "I, Diane  G. Galvin"
reporter_idx = prelim_docs[119].upper().find("I, DIANE G. GALVIN")
prelim_docs[119] = prelim_docs[119][:reporter_idx] # keep everything up to but not including the reporter information

In [239]:
# Check that the end matter was successfully removed from all transctiption documents
for i in docs_w_transcription_junk[:10]:
    print(f"Document {i} ends with: {prelim_docs[i][-100:]}") # check the last 100 characters of the document to confirm that the end matter was removed

Document 103 ends with: OWSIE : So moved . The meeting is adjourned . The meeting adjourned at approximately 7:45 p.m.  197 
Document 104 ends with:  HOWSIE : We're adjourned . (Whereupon , the hearing was concluded at approximately 7:22 p.m.)  203 
Document 105 ends with: ry one. Stay safe. See you in the new year. (The meeting concluded at approximately 7:40 p.m.)  224 
Document 115 ends with: rn . JUDGE HOWSIE : Thank you. I second . (Whereupon , the hearing was concluded at 6:49 p.m.)  180 
Document 116 ends with: E : Thank you. The meeting is adjourned . (Whereupon , the meeting was adjourned at 6:49 p.m.)  163 
Document 117 ends with: all.  166 MS. HALLAM : Great job, Terri . (Whereupon , the hearing was adjourned at 6:47 p.m.)  167 
Document 118 ends with: KRAUS : Motion to adjourn . (Whereupon , the meeting was concluded at approximately 7:45 p.m.)  171 
Document 119 ends with: he hearing was concluded at 6:45 p.m.)  157 COMMONWEALTH OF PENNSYLVANIA ) ss COUNTY OF ALLEGHENY ) 


### Final preprocessing with `spaCy`

This section includes removing stop words, tokenizing, and lemmatizing.

After initial exploration in n-grams and LDA, we found additional words and names that should be treated as stop words, which have been added below.

In [305]:
manual_stop_words = [
    # names as found during the n-gram analysis
    'judge court', 'judge joseph', 'judge judge', 'mr .', 'sheriff department',
    'sheriff kevin', 'sheriff office', 'sheriff william', 'warden address',
    'warden administrative', 'warden answer', 'warden chief', 'warden deputy',
    'warden good', 'warden jail', 'warden jason', 'warden judge', 'warden know',
    'warden like', 'warden long', 'warden respond', 'warden staff', 'warden state',
    'warden talk', 'warden think', 'warden update', 'warden want', 'warden warden',
    'warden work', 'warden ms.', 'warden provide', 'warden response', 'warden search',

    # fillers during conversations - should not be meaningful
    'know', 'think', 'want', 'okay', 'thank', 'like',
    'actually', 'thing', 'ask', 'look', 'say', 'yes', 'yeah', 'sure',
    'oh', 'sorry', 'maybe', 'lot', 
    'let', 'bring', 'tell', 'come', 'happen', 'hear',
    'speak', 'try', 'find', 'understand', 'able',
    'way', 'today', 'anybody', 'everybody', 'somebody', 'guy', 'folk',
    'far', 'past', 'long', 'different', 'specific', 'specifically',
    'currently', 'actually', 'forward', 'outside', 'available', 'little',
    'couple', 'ago', 'appreciate', 'mention', 'run', 'add', 'end', 'page',
    
    # some of the single letters are showing up as frequent words
    'e', 't', 's', 'o', 'n', 'r', 'h', 'w', 'c', 'd', 'l', 'g',
    'f', 'p', 'u', 'm', 'y', "'",
    
    # meeting formal words
    'meeting', 'motion', 'second', 'approve', 'adjourn', 'report',
    'member', 'present', 'minute', 'vote', 'board', 'committee',
    'comment', 'public', 'business', 'follow', 'address', 'discussion',
    'record', 'hold', 'begin', 'complete', 'submit', 'review', 'executive', 'president',
    
    # places/locations
    'allegheny', 'county', 'pennsylvania', 'pittsburgh',
    
    # names of specific board members
    'hallam', 'evashavik', 'howsie', 'beasom', 'lazzara', 'brinkman',
    'toma', 'bigley', 'innamorato', 'damick', 'wagner', 'moss',
    'klein', 'perkins', 'williams', 'clark', 'bethany', 'ludwig', 'mcdaniel', 'ron', 'dana', 'chuck', 'doug',
    'larry', 'pofi', 'acchs', 'martoni', 'madarino', 'catanese',
    'phillips', 'donna', 'jo', 'joe', 'orlando', 'harper', 'latoya',
    'mullen', 'fitzgerald', 'asturi', 'carol', 'hertz', 'mike',
    'gilmore', 'marion', 'chelsa', 'william', 'dilucente', '-dilucente', 'cashman',
    'walker', 'corizon', 'wingard', 'trevor', 'defazio', 'korinski',
    'dilucente', 'right', 'talk', 'man', 'hood', 'price', 'griffin', 'officer', 'connor', 'acj', 'mean', 'zak', 'person',
    'visit', 'beth', 'ali', 'kroll', 'davis', 'martin', 'judgeclark', "o'connor", 'kimberly', 'm.', 'gayle', 'elliott',

    # titles, prefixes, and other noticed words/characters with little or no meaning
    '.', '-', 'mr', 'ms', 'mr.', 'ms.', 'dr.', 'dr', 'inmate', 'jail', 'judge', 'warden', 'report', 'oversight', 'board',
    'conference', 'room', 'court', 'honorable', '00', '000', '000incarceratedindividual', '00am', '00o', '00p', '00pm',
    '01', '015', '02', '023', '03', '038', '04', '040', '048', '05', '050', '06', '069',
    'actions', 'mrs', 'following', 'duly', 'unanimously', 'approval', 'agreement', 'share', 'austin', 'party', 'respectfully', 
    'adjournment', 'secretary', 'parole', 'distribute', 'inform', 'mrs.', 'represent'
]

In [306]:
len(manual_stop_words)

274

In [241]:
import spacy
nlp = spacy.load('en_core_web_sm', disable = ['parser', 'ner']) # disable parser and NER so package runs faster

In [302]:
# Define function to keep only the lemmas we care about; this function removes stop words, tokenizes, and lemmatizes all at once
# This function is courtesy of class demos, with minor tweaks
def get_filtered_words(text):
    filtered_lemmas = []
    for token in nlp(text):
        lemma = token.lemma_.lower()
        if not (nlp.vocab[lemma].is_stop
                or lemma in manual_stop_words
                or token.pos_ == 'PUNCT'
                or token.pos_ == 'SPACE'
                or token.pos_ == 'SYM'
                or token.pos_ == 'X') and (
                    # Remove digits, unless that digit is a 4-digit year starting with "19" or "20" (e.g. 1995, 2020)
                    not lemma.isdigit()
                    or (lemma.isdigit() and len(lemma) == 4 and lemma.startswith(('19', '20')))
                ):
            filtered_lemmas.append(lemma)
    return filtered_lemmas

In [307]:
from tqdm import tqdm

documents = []
for prelim_doc in tqdm(prelim_docs): # use tqdm for progress monitoring; time varies depending on your machine
    documents.append(get_filtered_words(prelim_doc))

100%|██████████| 156/156 [01:28<00:00,  1.76it/s]


## Confirm Successful Preprocessing

Finally, we check whether the removal, tokenization, and lemmatization worked correctly by examining the first few documents.

In [308]:
print(documents[:2]) # check the first 2 documents to confirm that the cleaning worked as expected

[['monthly', 'thursday', 'october', '2012', 'house', '4:00', 'p.m.', 'chief', 'defender', 'rich', 'charles', 'sheriff', 'major', 'interested', 'discuss', 'request', 'discuss', 'role', 'concentrate', 'library', 'learning', 'teaching', 'read', 'provide', 'brochure', 'pa', 'prison', 'society', 'supreme', 'rule', 'state', 'pay', 'september', '2012', 'newspaper', 'confinement', 'confine', 'hour', 'solitary', 'confinement', 'solitary', 'confinement', 'finally', 'acting', 'stickman', 'permit', 'voting', 'absentee', 'ballot', 'misdemeanor', 'september', '2012', 'discuss', 'attend', 're-', 'entry', 'state', 'attend', 'sec', 'john', 'wetzel', 'chief', 'probation', 'potinger', 'david', 'hichton', 'u.s.', 'attorney', 'keynote', 'speaker', 'welcome', 'new', 'state', 'promotion', 'warren', 'deputy', 'services', 'program', 'services', 'old', 'naacp', 'floor', 'collect', 'absentee', 'ballot', 'inmates', 'yesterday', 'new', 'correctional', 'health', 'services', 'month', 'collaborative', 'providers', 'f

It appears our preprocessing has been successful. From here, we will save the preprocessed text and then analyze the board meeting minutes via n-grams and topic modeling in subsequent code files.

In [309]:
# Adapted from UDA Preprocessing 20 Newsgroups.ipynb
import pickle

# Creating file path to Text folder
path = OUT_DIR / "lemmatized_text.xz"

# Saving preprocessed text
with open(path, 'wb') as f:
    pickle.dump(documents, f)